In [1]:
# =========================
# BLOCK: CONFIGURATION AND IMPORTS
# =========================

from pathlib import Path
import re
import json
import shutil
import textwrap
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------
# Main configurable paths
# -------------------------
ROOT = Path("C:/punya qq/tugas-akhir-qq/code/V2/workspace_vastAI_v2")

OUTPUTS = ROOT / "outputs" / "outputs"
OUT_FINAL_RUNS = OUTPUTS / "final_runs"
OUT_FINAL_AGG = OUTPUTS / "final_aggregate"
OUT_FINAL_SELECTED = OUTPUTS / "final_selected"
OUT_APPENDIX = OUTPUTS / "appendix_ready"
OUT_POSTHOC = OUTPUTS / "posthoc_diagnostics"

DATA = ROOT / "data" / "data"
MANIFEST_VAST = DATA / "manifest_strict_vast.csv"

FINAL_RUNS_SUMMARY_PATH = OUT_FINAL_AGG / "final_runs_summary.csv"
OPTUNA_STUDY_CONFIG_PATH = OUTPUTS / "optuna_study" / "study_config.json"

# -------------------------
# Display control
# -------------------------
DISPLAY_PLOTS = False
FIG_DPI = 200

# -------------------------
# Expected final run seeds
# -------------------------
FINAL_RUN_SEEDS = [42, 52, 62, 72, 82, 92, 102, 112, 122, 132]

EXPECTED_FINAL_RUNS = [
    {
        "run_id": f"run_{idx:02d}_seed_{seed:03d}",
        "seed": seed,
        "run_dir": OUT_FINAL_RUNS / f"run_{idx:02d}_seed_{seed:03d}",
        "expected": True,
    }
    for idx, seed in enumerate(FINAL_RUN_SEEDS, start=1)
]

REQUIRED_HISTORY_COLUMNS = [
    "epoch",
    "train_loss",
    "val_mae_mean",
]

print("ROOT:", ROOT)
print("OUTPUTS:", OUTPUTS)
print("OUT_FINAL_RUNS:", OUT_FINAL_RUNS)
print("OUT_POSTHOC:", OUT_POSTHOC)
print("OUT_APPENDIX:", OUT_APPENDIX)

ROOT: C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2
OUTPUTS: C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs
OUT_FINAL_RUNS: C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\final_runs
OUT_POSTHOC: C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\posthoc_diagnostics
OUT_APPENDIX: C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\appendix_ready


In [2]:
# =========================
# BLOCK: HELPER FUNCTIONS
# =========================

def ensure_dir(path: Path) -> Path:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def save_csv(df: pd.DataFrame, path: Path, index: bool = False) -> Path:
    path = Path(path)
    ensure_dir(path.parent)
    df.to_csv(path, index=index)
    return path


def save_text(text: str, path: Path) -> Path:
    path = Path(path)
    ensure_dir(path.parent)
    path.write_text(str(text), encoding="utf-8")
    return path


def save_and_maybe_show(fig, path: Path) -> Path:
    path = Path(path)
    ensure_dir(path.parent)
    fig.tight_layout()
    fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
    
    if DISPLAY_PLOTS:
        plt.show()
    
    plt.close(fig)
    return path


def safe_read_csv(path: Path, required: bool = False, **kwargs):
    path = Path(path)
    
    if not path.exists():
        msg = f"[SKIP] File tidak ditemukan: {path}"
        if required:
            raise FileNotFoundError(msg)
        print(msg)
        return None
    
    try:
        return pd.read_csv(path, **kwargs)
    except Exception as exc:
        msg = f"[ERROR] Gagal membaca CSV: {path}\nAlasan: {exc}"
        if required:
            raise RuntimeError(msg) from exc
        print(msg)
        return None


def pick_col(df: pd.DataFrame, candidates):
    if df is None:
        return None
    
    lower_to_original = {str(col).lower(): col for col in df.columns}
    
    for cand in candidates:
        cand_lower = str(cand).lower()
        if cand_lower in lower_to_original:
            return lower_to_original[cand_lower]
    
    return None


def parse_seed_from_run_id(run_id: str):
    match = re.search(r"seed[_-]?(\d+)", str(run_id))
    if match:
        return int(match.group(1))
    return None


def row_to_lower_dict(row) -> dict:
    return {str(k).lower(): v for k, v in dict(row).items()}


def get_value(row_dict: dict, candidates, default=np.nan):
    if row_dict is None:
        return default
    
    for cand in candidates:
        key = str(cand).lower()
        if key in row_dict and pd.notna(row_dict[key]):
            return row_dict[key]
    
    return default


def get_numeric_value(row_dict: dict, candidates, default=np.nan):
    value = get_value(row_dict, candidates, default=default)
    
    if pd.isna(value):
        return default
    
    try:
        return pd.to_numeric(value)
    except Exception:
        return default


def normalize_epoch_value(value):
    if pd.isna(value):
        return np.nan
    
    try:
        return int(float(value))
    except Exception:
        return np.nan


def find_summary_row(summary_df: pd.DataFrame, run_id: str, seed):
    if summary_df is None or len(summary_df) == 0:
        return {}
    
    run_col = pick_col(summary_df, ["run_id", "run", "run_name", "run_dir", "folder", "experiment_id"])
    seed_col = pick_col(summary_df, ["seed", "random_seed"])
    
    if run_col is not None:
        mask = summary_df[run_col].astype(str).eq(str(run_id))
        if mask.any():
            return row_to_lower_dict(summary_df.loc[mask].iloc[0])
        
        # Antisipasi jika kolom run menyimpan path, bukan hanya nama folder
        mask_contains = summary_df[run_col].astype(str).str.contains(str(run_id), regex=False, na=False)
        if mask_contains.any():
            return row_to_lower_dict(summary_df.loc[mask_contains].iloc[0])
    
    if seed_col is not None and seed is not None:
        seed_series = pd.to_numeric(summary_df[seed_col], errors="coerce")
        mask = seed_series.eq(int(seed))
        if mask.any():
            return row_to_lower_dict(summary_df.loc[mask].iloc[0])
    
    return {}


def get_row_at_or_nearest_epoch(history_df: pd.DataFrame, epoch_value):
    if history_df is None or len(history_df) == 0:
        return None
    
    if pd.isna(epoch_value):
        idx = history_df["val_loss"].idxmin()
        return history_df.loc[idx]
    
    epoch_value = float(epoch_value)
    idx = (history_df["epoch"].astype(float) - epoch_value).abs().idxmin()
    return history_df.loc[idx]


def format_float(value, digits=6):
    if pd.isna(value):
        return "NA"
    return f"{float(value):.{digits}f}"

In [3]:
# =========================
# BLOCK: VALIDATE PATHS AND LOAD FINAL RUN SUMMARY
# =========================

if not ROOT.exists():
    raise FileNotFoundError(
        f"""
ROOT tidak ditemukan: {ROOT}

Silakan sesuaikan variabel ROOT di cell konfigurasi.
Notebook ini tidak melakukan training ulang dan hanya membaca output yang sudah ada.
        """.strip()
    )

if not OUTPUTS.exists():
    raise FileNotFoundError(
        f"""
Folder outputs tidak ditemukan: {OUTPUTS}

Pastikan notebook ini dijalankan di environment yang sama dengan output eksperimen utama.
        """.strip()
    )

ensure_dir(OUT_POSTHOC)
ensure_dir(OUT_APPENDIX)

final_runs_summary_df = safe_read_csv(FINAL_RUNS_SUMMARY_PATH, required=False)

if final_runs_summary_df is not None:
    print(f"final_runs_summary.csv terbaca: {FINAL_RUNS_SUMMARY_PATH}")
    display(final_runs_summary_df.head())
else:
    print("final_runs_summary.csv tidak ditemukan. Analisis tetap berjalan dengan informasi dari folder run.")

final_runs_summary.csv terbaca: C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\final_aggregate\final_runs_summary.csv


,run_id,seed,best_epoch,stop_epoch,best_val_S,test_MAE_mean,test_RMSE_mean,test_R2_mean,test_Acc_mean
0,run_01_seed_042,42,24,29,0.901935,0.100190,0.126587,0.303067,0.899810
1,run_02_seed_052,52,20,25,0.901672,0.100338,0.126723,0.301679,0.899662
2,run_03_seed_062,62,21,26,0.901894,0.100174,0.126508,0.304086,0.899826
3,run_04_seed_072,72,23,28,0.901856,0.100350,0.126765,0.300896,0.899650
4,run_05_seed_082,82,24,29,0.901691,0.100302,0.126842,0.300112,0.899698


In [4]:
# =========================
# BLOCK: DISCOVER FINAL RUN DIRECTORIES AND HISTORY FILES
# =========================

run_records = {}

# Masukkan run yang diharapkan berdasarkan fixed distinct seeds
for rec in EXPECTED_FINAL_RUNS:
    run_records[rec["run_id"]] = rec.copy()

# Tambahkan run lain yang ditemukan otomatis di folder final_runs
if OUT_FINAL_RUNS.exists():
    for run_dir in sorted(OUT_FINAL_RUNS.glob("run_*")):
        if not run_dir.is_dir():
            continue
        
        run_id = run_dir.name
        seed = parse_seed_from_run_id(run_id)
        
        if run_id not in run_records:
            run_records[run_id] = {
                "run_id": run_id,
                "seed": seed,
                "run_dir": run_dir,
                "expected": False,
            }

run_index_rows = []

for run_id, rec in sorted(run_records.items()):
    run_dir = Path(rec["run_dir"])
    history_path = run_dir / "history.csv"
    
    seed = rec.get("seed")
    if seed is None:
        seed = parse_seed_from_run_id(run_id)
    
    run_index_rows.append({
        "run_id": run_id,
        "seed": seed,
        "run_dir": str(run_dir),
        "history_path": str(history_path),
        "history_exists": history_path.exists(),
        "expected": rec.get("expected", False),
    })

run_index_df = pd.DataFrame(run_index_rows)
run_index_path = save_csv(run_index_df, OUT_POSTHOC / "final_run_history_index.csv")

missing_history_df = run_index_df[~run_index_df["history_exists"]].copy()
missing_history_path = OUT_POSTHOC / "missing_history_files.txt"

if len(missing_history_df) > 0:
    missing_text = "\n".join(missing_history_df["history_path"].tolist())
else:
    missing_text = "Tidak ada history.csv yang missing dari run yang terindeks."

save_text(missing_text, missing_history_path)

print(f"Index history disimpan ke: {run_index_path}")
print(f"Catatan missing history disimpan ke: {missing_history_path}")
display(run_index_df)

Index history disimpan ke: C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\posthoc_diagnostics\final_run_history_index.csv
Catatan missing history disimpan ke: C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\posthoc_diagnostics\missing_history_files.txt


,run_id,seed,run_dir,history_path,history_exists,expected
0,run_01_seed_042,42,C:\punya qq\tugas-akhir-qq\code\V2\workspace_v...,C:\punya qq\tugas-akhir-qq\code\V2\workspace_v...,True,True
1,run_02_seed_052,52,C:\punya qq\tugas-akhir-qq\code\V2\workspace_v...,C:\punya qq\tugas-akhir-qq\code\V2\workspace_v...,True,True
2,run_03_seed_062,62,C:\punya qq\tugas-akhir-qq\code\V2\workspace_v...,C:\punya qq\tugas-akhir-qq\code\V2\workspace_v...,True,True
3,run_04_seed_072,72,C:\punya qq\tugas-akhir-qq\code\V2\workspace_v...,C:\punya qq\tugas-akhir-qq\code\V2\workspace_v...,True,True
4,run_05_seed_082,82,C:\punya qq\tugas-akhir-qq\code\V2\workspace_v...,C:\punya qq\tugas-akhir-qq\code\V2\workspace_v...,True,True
5,run_06_seed_092,92,C:\punya qq\tugas-akhir-qq\code\V2\workspace_v...,C:\punya qq\tugas-akhir-qq\code\V2\workspace_v...,True,True
6,run_07_seed_102,102,C:\punya qq\tugas-akhir-qq\code\V2\workspace_v...,C:\punya qq\tugas-akhir-qq\code\V2\workspace_v...,True,True
7,run_08_seed_112,112,C:\punya qq\tugas-akhir-qq\code\V2\workspace_v...,C:\punya qq\tugas-akhir-qq\code\V2\workspace_v...,True,True
8,run_09_seed_122,122,C:\punya qq\tugas-akhir-qq\code\V2\workspace_v...,C:\punya qq\tugas-akhir-qq\code\V2\workspace_v...,True,True
9,run_10_seed_132,132,C:\punya qq\tugas-akhir-qq\code\V2\workspace_v...,C:\punya qq\tugas-akhir-qq\code\V2\workspace_v...,True,True


In [5]:
# =========================
# BLOCK: LOAD HISTORIES AND ADD VAL LOSS DIAGNOSTIC COLUMNS
# =========================

RUN_HISTORIES = {}
RUN_META = {}
invalid_history_notes = []

for _, row in run_index_df.iterrows():
    if not bool(row["history_exists"]):
        continue
    
    run_id = row["run_id"]
    seed = row["seed"]
    run_dir = Path(row["run_dir"])
    history_path = Path(row["history_path"])
    
    history_df = safe_read_csv(history_path, required=False)
    
    if history_df is None:
        invalid_history_notes.append(f"{run_id}: gagal membaca {history_path}")
        continue
    
    missing_cols = [col for col in REQUIRED_HISTORY_COLUMNS if col not in history_df.columns]
    if missing_cols:
        invalid_history_notes.append(
            f"{run_id}: kolom wajib tidak ditemukan {missing_cols} pada {history_path}"
        )
        continue
    
    # Ambil metadata dari final_runs_summary.csv jika tersedia
    summary_row = find_summary_row(final_runs_summary_df, run_id=run_id, seed=seed)
    
    seed_from_summary = get_numeric_value(summary_row, ["seed", "random_seed"], default=np.nan)
    if pd.notna(seed_from_summary):
        seed = int(seed_from_summary)
    
    best_epoch = normalize_epoch_value(
        get_numeric_value(
            summary_row,
            ["best_epoch", "best_val_epoch", "epoch_best", "selected_epoch"],
            default=np.nan,
        )
    )
    
    stop_epoch = normalize_epoch_value(
        get_numeric_value(
            summary_row,
            ["stop_epoch", "stopped_epoch", "early_stop_epoch", "last_epoch"],
            default=np.nan,
        )
    )
    
    # Pastikan numerik
    history_df = history_df.copy()
    
    numeric_candidates = [
        "epoch",
        "train_loss",
        "val_mae_mean",
        "val_rmse_mean",
        "val_r2_mean",
        "val_acc_mean",
        "val_S",
        "val_mae_extraversion",
        "val_mae_neuroticism",
        "val_mae_agreeableness",
        "val_mae_conscientiousness",
        "val_mae_openness",
    ]
    
    for col in numeric_candidates:
        if col in history_df.columns:
            history_df[col] = pd.to_numeric(history_df[col], errors="coerce")
    
    history_df = history_df.dropna(subset=["epoch", "train_loss", "val_mae_mean"]).copy()
    history_df["epoch"] = history_df["epoch"].astype(int)
    history_df = history_df.sort_values("epoch").reset_index(drop=True)
    
    # Kolom diagnostik utama
    history_df["run_id"] = run_id
    history_df["seed"] = seed
    history_df["val_loss"] = history_df["val_mae_mean"]
    history_df["train_val_gap"] = history_df["val_loss"] - history_df["train_loss"]
    
    # Susun kolom minimal di depan
    front_cols = [
        "run_id",
        "seed",
        "epoch",
        "train_loss",
        "val_loss",
        "val_mae_mean",
        "val_rmse_mean",
        "val_r2_mean",
        "val_S",
        "train_val_gap",
    ]
    
    existing_front_cols = [col for col in front_cols if col in history_df.columns]
    remaining_cols = [col for col in history_df.columns if col not in existing_front_cols]
    history_out = history_df[existing_front_cols + remaining_cols].copy()
    
    out_path = run_dir / "history_with_val_loss.csv"
    save_csv(history_out, out_path)
    
    RUN_HISTORIES[run_id] = history_out
    RUN_META[run_id] = {
        "run_id": run_id,
        "seed": seed,
        "run_dir": run_dir,
        "history_path": history_path,
        "history_with_val_loss_path": out_path,
        "best_epoch": best_epoch,
        "stop_epoch": stop_epoch,
    }

invalid_history_path = OUT_POSTHOC / "invalid_history_files.txt"

if invalid_history_notes:
    save_text("\n".join(invalid_history_notes), invalid_history_path)
else:
    save_text("Tidak ada history.csv invalid.", invalid_history_path)

if len(RUN_HISTORIES) == 0:
    raise FileNotFoundError(
        """
Tidak ada history.csv valid yang berhasil dibaca.

Notebook ini tidak bisa melanjutkan diagnostic loss tanpa minimal satu history.csv.
Cek file:
- outputs/posthoc_diagnostics/final_run_history_index.csv
- outputs/posthoc_diagnostics/missing_history_files.txt
- outputs/posthoc_diagnostics/invalid_history_files.txt
        """.strip()
    )

print(f"Jumlah run valid terbaca: {len(RUN_HISTORIES)}")
print(f"Catatan invalid history: {invalid_history_path}")

for run_id, meta in RUN_META.items():
    print(f"{run_id}: saved -> {meta['history_with_val_loss_path']}")

Jumlah run valid terbaca: 10
Catatan invalid history: C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\posthoc_diagnostics\invalid_history_files.txt
run_01_seed_042: saved -> C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\final_runs\run_01_seed_042\history_with_val_loss.csv
run_02_seed_052: saved -> C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\final_runs\run_02_seed_052\history_with_val_loss.csv
run_03_seed_062: saved -> C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\final_runs\run_03_seed_062\history_with_val_loss.csv
run_04_seed_072: saved -> C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\final_runs\run_04_seed_072\history_with_val_loss.csv
run_05_seed_082: saved -> C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\final_runs\run_05_seed_082\history_with_val_loss.csv
run_06_seed_092: saved -> C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v

In [6]:
# =========================
# BLOCK: PER-RUN TRAIN VS VALIDATION LOSS CURVES
# =========================

per_run_plot_paths = []

for run_id, history_df in RUN_HISTORIES.items():
    meta = RUN_META[run_id]
    seed = meta.get("seed")
    best_epoch = meta.get("best_epoch")
    run_dir = Path(meta["run_dir"])
    
    title_seed = f"seed={seed}" if seed is not None and pd.notna(seed) else "seed=NA"
    
    # -------------------------
    # Plot train_loss vs val_loss
    # -------------------------
    fig, ax = plt.subplots(figsize=(8, 5))
    
    ax.plot(
        history_df["epoch"],
        history_df["train_loss"],
        label="train_loss (L1/MAE, train mode)",
        linewidth=2,
    )
    
    ax.plot(
        history_df["epoch"],
        history_df["val_loss"],
        label="val_loss = val_mae_mean",
        linewidth=2,
        linestyle="--",
    )
    
    if pd.notna(best_epoch):
        ax.axvline(
            int(best_epoch),
            linestyle=":",
            linewidth=1.5,
            label=f"best_epoch={int(best_epoch)}",
        )
    
    ax.set_title(f"Train vs Validation Loss - {run_id} ({title_seed})")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss / MAE")
    ax.grid(True, alpha=0.3)
    ax.legend()
    
    out_curve_path = run_dir / "curve_train_vs_val_loss.png"
    per_run_plot_paths.append(save_and_maybe_show(fig, out_curve_path))
    
    # -------------------------
    # Plot gap
    # -------------------------
    fig, ax = plt.subplots(figsize=(8, 5))
    
    ax.plot(
        history_df["epoch"],
        history_df["train_val_gap"],
        label="train_val_gap = val_loss - train_loss",
        linewidth=2,
    )
    
    ax.axhline(0, linestyle=":", linewidth=1.5)
    
    if pd.notna(best_epoch):
        ax.axvline(
            int(best_epoch),
            linestyle=":",
            linewidth=1.5,
            label=f"best_epoch={int(best_epoch)}",
        )
    
    ax.set_title(f"Train-Validation Gap - {run_id}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("val_loss - train_loss")
    ax.grid(True, alpha=0.3)
    ax.legend()
    
    out_gap_path = run_dir / "curve_train_val_gap.png"
    per_run_plot_paths.append(save_and_maybe_show(fig, out_gap_path))

print("Plot per-run selesai dibuat:")
for path in per_run_plot_paths:
    print("-", path)

Plot per-run selesai dibuat:
- C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\final_runs\run_01_seed_042\curve_train_vs_val_loss.png
- C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\final_runs\run_01_seed_042\curve_train_val_gap.png
- C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\final_runs\run_02_seed_052\curve_train_vs_val_loss.png
- C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\final_runs\run_02_seed_052\curve_train_val_gap.png
- C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\final_runs\run_03_seed_062\curve_train_vs_val_loss.png
- C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\final_runs\run_03_seed_062\curve_train_val_gap.png
- C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\final_runs\run_04_seed_072\curve_train_vs_val_loss.png
- C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\final_runs\run

In [7]:
# =========================
# BLOCK: DIAGNOSIS LOGIC FOR OVERFITTING AND UNDERFITTING
# =========================

# Threshold dibuat sederhana dan konservatif.
# Nilai ini bisa disesuaikan jika dosen meminta kriteria yang lebih spesifik.
SMALL_IMPROVEMENT_THRESHOLD = 0.003
GAP_INCREASE_THRESHOLD = 0.010
HIGH_LOSS_THRESHOLD = 0.120
STABLE_GAP_THRESHOLD = 0.020
VAL_DEGRADATION_THRESHOLD = 0.003


def diagnose_run(history_df: pd.DataFrame) -> str:
    h = history_df.sort_values("epoch").reset_index(drop=True).copy()
    
    first_train = float(h["train_loss"].iloc[0])
    first_val = float(h["val_loss"].iloc[0])
    final_train = float(h["train_loss"].iloc[-1])
    final_val = float(h["val_loss"].iloc[-1])
    final_gap = float(h["train_val_gap"].iloc[-1])
    
    min_train = float(h["train_loss"].min())
    min_val_idx = h["val_loss"].idxmin()
    min_val_row = h.loc[min_val_idx]
    
    min_val = float(min_val_row["val_loss"])
    train_at_min_val = float(min_val_row["train_loss"])
    gap_at_min_val = float(min_val_row["train_val_gap"])
    epoch_min_val = int(min_val_row["epoch"])
    
    train_improvement = first_train - min_train
    val_improvement = first_val - min_val
    
    val_rose_after_min = final_val > (min_val + VAL_DEGRADATION_THRESHOLD)
    train_continued_down_after_min = final_train < (train_at_min_val - SMALL_IMPROVEMENT_THRESHOLD)
    gap_increased_after_min = final_gap > (gap_at_min_val + GAP_INCREASE_THRESHOLD)
    
    both_losses_high = final_train > HIGH_LOSS_THRESHOLD and final_val > HIGH_LOSS_THRESHOLD
    little_improvement = (
        train_improvement < SMALL_IMPROVEMENT_THRESHOLD
        and val_improvement < SMALL_IMPROVEMENT_THRESHOLD
    )
    
    gap_stable = abs(final_gap) <= STABLE_GAP_THRESHOLD
    val_near_best = final_val <= (min_val + VAL_DEGRADATION_THRESHOLD)
    
    if val_rose_after_min and train_continued_down_after_min and gap_increased_after_min:
        return (
            "indikasi overfitting ringan setelah epoch terbaik / early stopping bekerja; "
            "val_loss minimum terjadi pada epoch "
            f"{epoch_min_val}, lalu val_loss meningkat sementara train_loss masih menurun."
        )
    
    if both_losses_high and little_improvement:
        return (
            "indikasi underfitting; train_loss dan val_loss relatif tinggi serta tidak menunjukkan "
            "perbaikan yang besar sepanjang epoch."
        )
    
    if gap_stable and val_near_best:
        return (
            "stabil / tidak menunjukkan overfitting berat; train_loss dan val_loss bergerak relatif "
            "konvergen dengan gap yang tidak membesar ekstrem."
        )
    
    if val_rose_after_min and not gap_increased_after_min:
        return (
            "indikasi fluktuasi validasi setelah epoch terbaik, tetapi gap train-validation tidak "
            "membesar ekstrem sehingga belum cukup kuat untuk disebut overfitting berat."
        )
    
    return (
        "perlu interpretasi hati-hati; pola loss tidak menunjukkan underfitting jelas maupun "
        "overfitting berat berdasarkan aturan diagnostik sederhana ini."
    )

In [8]:
# =========================
# BLOCK: BUILD LOSS DIAGNOSTIC SUMMARY PER RUN
# =========================

summary_rows = []

for run_id, history_df in RUN_HISTORIES.items():
    h = history_df.sort_values("epoch").reset_index(drop=True).copy()
    meta = RUN_META[run_id]
    
    seed = meta.get("seed")
    best_epoch = meta.get("best_epoch")
    stop_epoch = meta.get("stop_epoch")
    
    final_epoch = int(h["epoch"].max())
    
    min_train_loss = float(h["train_loss"].min())
    min_val_idx = h["val_loss"].idxmin()
    min_val_row = h.loc[min_val_idx]
    
    min_val_loss = float(min_val_row["val_loss"])
    epoch_min_val_loss = int(min_val_row["epoch"])
    
    # Jika best_epoch tidak tersedia dari final_runs_summary.csv,
    # gunakan epoch dengan val_loss minimum sebagai fallback diagnostik.
    if pd.isna(best_epoch):
        best_epoch = epoch_min_val_loss
    
    if pd.isna(stop_epoch):
        stop_epoch = final_epoch
    
    best_row = get_row_at_or_nearest_epoch(h, best_epoch)
    final_row = h.iloc[-1]
    
    train_loss_at_best_epoch = float(best_row["train_loss"])
    val_loss_at_best_epoch = float(best_row["val_loss"])
    gap_at_best_epoch = float(best_row["train_val_gap"])
    
    final_train_loss = float(final_row["train_loss"])
    final_val_loss = float(final_row["val_loss"])
    final_gap = float(final_row["train_val_gap"])
    
    mean_gap = float(h["train_val_gap"].mean())
    max_gap = float(h["train_val_gap"].max())
    min_gap = float(h["train_val_gap"].min())
    
    diagnosis = diagnose_run(h)
    
    summary_rows.append({
        "run_id": run_id,
        "seed": seed,
        "best_epoch": int(best_epoch) if pd.notna(best_epoch) else np.nan,
        "stop_epoch": int(stop_epoch) if pd.notna(stop_epoch) else np.nan,
        "final_epoch": final_epoch,
        "min_train_loss": min_train_loss,
        "min_val_loss": min_val_loss,
        "epoch_min_val_loss": epoch_min_val_loss,
        "train_loss_at_best_epoch": train_loss_at_best_epoch,
        "val_loss_at_best_epoch": val_loss_at_best_epoch,
        "gap_at_best_epoch": gap_at_best_epoch,
        "final_train_loss": final_train_loss,
        "final_val_loss": final_val_loss,
        "final_gap": final_gap,
        "mean_gap": mean_gap,
        "max_gap": max_gap,
        "min_gap": min_gap,
        "diagnosis": diagnosis,
    })

loss_diagnostic_summary_df = pd.DataFrame(summary_rows)
loss_diagnostic_summary_path = save_csv(
    loss_diagnostic_summary_df,
    OUT_POSTHOC / "loss_diagnostic_summary_per_run.csv",
)

print(f"Summary per run disimpan ke: {loss_diagnostic_summary_path}")
display(loss_diagnostic_summary_df)

Summary per run disimpan ke: C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\posthoc_diagnostics\loss_diagnostic_summary_per_run.csv


,run_id,seed,best_epoch,stop_epoch,final_epoch,min_train_loss,min_val_loss,epoch_min_val_loss,train_loss_at_best_epoch,val_loss_at_best_epoch,gap_at_best_epoch,final_train_loss,final_val_loss,final_gap,mean_gap,max_gap,min_gap,diagnosis
0,run_01_seed_042,42,24,29,29,0.093535,0.098065,24,0.094404,0.098065,0.003661,0.093535,0.098203,0.004668,0.002612,0.004776,-0.003323,stabil / tidak menunjukkan overfitting berat; ...
1,run_02_seed_052,52,20,25,25,0.093958,0.098328,20,0.094986,0.098328,0.003343,0.093958,0.098632,0.004674,0.002177,0.004674,-0.004294,stabil / tidak menunjukkan overfitting berat; ...
2,run_03_seed_062,62,21,26,26,0.094005,0.098106,21,0.094607,0.098106,0.003500,0.094053,0.098644,0.004591,0.002348,0.005687,-0.003784,stabil / tidak menunjukkan overfitting berat; ...
3,run_04_seed_072,72,23,28,28,0.093795,0.098144,23,0.094336,0.098144,0.003808,0.093826,0.098176,0.004350,0.002477,0.005262,-0.003428,stabil / tidak menunjukkan overfitting berat; ...
4,run_05_seed_082,82,24,29,29,0.093684,0.098309,24,0.094085,0.098309,0.004224,0.093724,0.098406,0.004682,0.002551,0.004966,-0.003943,stabil / tidak menunjukkan overfitting berat; ...
5,run_06_seed_092,92,19,24,24,0.094179,0.098252,19,0.094785,0.098252,0.003467,0.094179,0.098791,0.004612,0.002258,0.005233,-0.003659,stabil / tidak menunjukkan overfitting berat; ...
6,run_07_seed_102,102,17,22,22,0.094217,0.098126,17,0.094730,0.098126,0.003397,0.094343,0.098465,0.004123,0.001873,0.004325,-0.004179,stabil / tidak menunjukkan overfitting berat; ...
7,run_08_seed_112,112,17,22,22,0.094291,0.098190,17,0.095250,0.098190,0.002940,0.094291,0.098677,0.004386,0.001900,0.004386,-0.004000,stabil / tidak menunjukkan overfitting berat; ...
8,run_09_seed_122,122,29,34,34,0.093362,0.098107,29,0.093732,0.098107,0.004375,0.093365,0.098485,0.005120,0.002961,0.005555,-0.003302,stabil / tidak menunjukkan overfitting berat; ...
9,run_10_seed_132,132,29,34,34,0.093282,0.098089,29,0.093886,0.098089,0.004203,0.093577,0.098120,0.004544,0.002735,0.005203,-0.004072,stabil / tidak menunjukkan overfitting berat; ...


In [9]:
# =========================
# BLOCK: AGGREGATE ALL RUN HISTORIES BY EPOCH
# =========================

all_runs_history_df = pd.concat(RUN_HISTORIES.values(), ignore_index=True)
all_runs_history_path = save_csv(
    all_runs_history_df,
    OUT_POSTHOC / "all_runs_history_with_val_loss.csv",
)

aggregate_loss_curve_df = (
    all_runs_history_df
    .groupby("epoch", as_index=False)
    .agg(
        n_runs=("run_id", "nunique"),
        train_loss_mean=("train_loss", "mean"),
        train_loss_std=("train_loss", "std"),
        val_loss_mean=("val_loss", "mean"),
        val_loss_std=("val_loss", "std"),
        gap_mean=("train_val_gap", "mean"),
        gap_std=("train_val_gap", "std"),
    )
)

aggregate_loss_curve_path = save_csv(
    aggregate_loss_curve_df,
    OUT_POSTHOC / "aggregate_loss_curve_by_epoch.csv",
)

print(f"All runs history disimpan ke: {all_runs_history_path}")
print(f"Aggregate curve by epoch disimpan ke: {aggregate_loss_curve_path}")
display(aggregate_loss_curve_df.head())

All runs history disimpan ke: C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\posthoc_diagnostics\all_runs_history_with_val_loss.csv
Aggregate curve by epoch disimpan ke: C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\posthoc_diagnostics\aggregate_loss_curve_by_epoch.csv


,epoch,n_runs,train_loss_mean,train_loss_std,val_loss_mean,val_loss_std,gap_mean,gap_std
0,1,10,0.122867,0.000804,0.119069,0.000495,-0.003798,0.000358
1,2,10,0.117736,0.000279,0.115127,0.000139,-0.002609,0.000200
2,3,10,0.113605,0.000271,0.111223,0.000300,-0.002382,0.000161
3,4,10,0.107882,0.000361,0.106092,0.000358,-0.001790,0.000283
4,5,10,0.103445,0.000174,0.103617,0.000462,0.000172,0.000479


In [10]:
# =========================
# BLOCK: AGGREGATE TRAIN VS VALIDATION LOSS MEAN STD PLOT
# =========================

plot_df = aggregate_loss_curve_df.sort_values("epoch").copy()

x = plot_df["epoch"].to_numpy(dtype=float)

train_mean = plot_df["train_loss_mean"].to_numpy(dtype=float)
train_std = plot_df["train_loss_std"].fillna(0).to_numpy(dtype=float)

val_mean = plot_df["val_loss_mean"].to_numpy(dtype=float)
val_std = plot_df["val_loss_std"].fillna(0).to_numpy(dtype=float)

fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(x, train_mean, label="Mean train_loss", linewidth=2)
ax.fill_between(
    x,
    train_mean - train_std,
    train_mean + train_std,
    alpha=0.15,
    label="Train ± std",
)

ax.plot(x, val_mean, label="Mean val_loss = val_mae_mean", linewidth=2, linestyle="--")
ax.fill_between(
    x,
    val_mean - val_std,
    val_mean + val_std,
    alpha=0.15,
    label="Validation ± std",
)

ax.set_title("Aggregate Train vs Validation Loss Across Final Runs")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss / MAE")
ax.grid(True, alpha=0.3)
ax.legend()

aggregate_train_vs_val_plot_path = save_and_maybe_show(
    fig,
    OUT_POSTHOC / "aggregate_train_vs_val_loss_mean_std.png",
)

print(f"Plot aggregate train vs val loss disimpan ke: {aggregate_train_vs_val_plot_path}")

Plot aggregate train vs val loss disimpan ke: C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\posthoc_diagnostics\aggregate_train_vs_val_loss_mean_std.png


In [11]:
# =========================
# BLOCK: AGGREGATE TRAIN VALIDATION GAP MEAN STD PLOT
# =========================

plot_df = aggregate_loss_curve_df.sort_values("epoch").copy()

x = plot_df["epoch"].to_numpy(dtype=float)
gap_mean = plot_df["gap_mean"].to_numpy(dtype=float)
gap_std = plot_df["gap_std"].fillna(0).to_numpy(dtype=float)

fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(x, gap_mean, label="Mean train_val_gap", linewidth=2)
ax.fill_between(
    x,
    gap_mean - gap_std,
    gap_mean + gap_std,
    alpha=0.15,
    label="Gap ± std",
)

ax.axhline(0, linestyle=":", linewidth=1.5)

ax.set_title("Aggregate Train-Validation Gap Across Final Runs")
ax.set_xlabel("Epoch")
ax.set_ylabel("val_loss - train_loss")
ax.grid(True, alpha=0.3)
ax.legend()

aggregate_gap_plot_path = save_and_maybe_show(
    fig,
    OUT_POSTHOC / "aggregate_train_val_gap_mean_std.png",
)

print(f"Plot aggregate gap disimpan ke: {aggregate_gap_plot_path}")

Plot aggregate gap disimpan ke: C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\posthoc_diagnostics\aggregate_train_val_gap_mean_std.png


In [12]:
# =========================
# BLOCK: ALL RUNS TRAIN VS VALIDATION LOSS OVERLAY
# =========================

fig, ax = plt.subplots(figsize=(10, 6))

for idx, (run_id, history_df) in enumerate(RUN_HISTORIES.items()):
    h = history_df.sort_values("epoch").copy()
    
    ax.plot(
        h["epoch"],
        h["train_loss"],
        linewidth=1.2,
        alpha=0.35,
        label="train_loss" if idx == 0 else None,
    )
    
    ax.plot(
        h["epoch"],
        h["val_loss"],
        linewidth=1.2,
        alpha=0.35,
        linestyle="--",
        label="val_loss" if idx == 0 else None,
    )

ax.set_title("Overlay Train vs Validation Loss Across Final Runs")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss / MAE")
ax.grid(True, alpha=0.3)
ax.legend()

overlay_plot_path = save_and_maybe_show(
    fig,
    OUT_POSTHOC / "all_runs_train_vs_val_loss_overlay.png",
)

print(f"Plot overlay semua run disimpan ke: {overlay_plot_path}")

Plot overlay semua run disimpan ke: C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\posthoc_diagnostics\all_runs_train_vs_val_loss_overlay.png


In [13]:
# =========================
# BLOCK: AUTOMATIC INDONESIAN INTERPRETATION TEXT
# =========================

mean_gap_at_best = loss_diagnostic_summary_df["gap_at_best_epoch"].mean()
mean_final_gap = loss_diagnostic_summary_df["final_gap"].mean()
mean_min_val_loss = loss_diagnostic_summary_df["min_val_loss"].mean()
mean_final_val_loss = loss_diagnostic_summary_df["final_val_loss"].mean()

diagnosis_counts = loss_diagnostic_summary_df["diagnosis"].value_counts()

overfit_like_count = loss_diagnostic_summary_df["diagnosis"].str.contains(
    "overfitting ringan", case=False, na=False
).sum()

underfit_like_count = loss_diagnostic_summary_df["diagnosis"].str.contains(
    "underfitting", case=False, na=False
).sum()

if overfit_like_count == 0 and underfit_like_count == 0:
    general_conclusion = (
        "Secara umum, pola loss pada 10 run tidak menunjukkan indikasi overfitting berat "
        "maupun underfitting yang jelas berdasarkan aturan diagnostik sederhana yang digunakan."
    )
elif overfit_like_count > 0 and overfit_like_count < len(loss_diagnostic_summary_df):
    general_conclusion = (
        "Sebagian run menunjukkan indikasi overfitting ringan setelah epoch terbaik, "
        "namun pola tersebut tidak muncul secara dominan pada seluruh run."
    )
elif overfit_like_count == len(loss_diagnostic_summary_df):
    general_conclusion = (
        "Mayoritas atau seluruh run menunjukkan indikasi overfitting ringan setelah epoch terbaik. "
        "Hal ini tetap perlu dibaca hati-hati karena early stopping memang dirancang untuk memilih checkpoint terbaik sebelum degradasi validasi berlanjut."
    )
elif underfit_like_count > 0:
    general_conclusion = (
        "Terdapat sebagian run dengan indikasi underfitting, tetapi interpretasi tetap perlu "
        "dibandingkan dengan kurva validasi dan metrik test akhir."
    )
else:
    general_conclusion = (
        "Pola loss perlu dibaca hati-hati karena tidak seluruh run mengikuti pola diagnostik yang sama."
    )

interpretation_text = f"""
Interpretasi Diagnostik Train Loss dan Validation Loss

Notebook pelengkap ini dibuat sebagai analisis post-hoc terhadap output eksperimen WavLM + LoRA pada strict split. Notebook ini tidak melakukan training ulang, tidak menjalankan Optuna ulang, dan tidak mengubah hasil eksperimen utama. Analisis dilakukan dengan membaca file history.csv dari setiap final run.

Pada notebook training utama, nilai train_loss dihitung menggunakan fungsi F.l1_loss(yhat, y, reduction="mean"). Dengan demikian, train_loss yang tersimpan pada history.csv merepresentasikan L1 loss atau MAE pada data train selama proses training. Sementara itu, metrik validasi dihitung melalui fungsi predict_and_evaluate(), yang menghasilkan val_mae_mean, val_rmse_mean, val_r2_mean, val_acc_mean, val_S, serta metrik per trait. Untuk kebutuhan diagnostik loss curve, notebook ini mendefinisikan val_loss sebagai alias dari val_mae_mean. Dengan kata lain, val_loss pada notebook pelengkap ini bukan MSE loss, melainkan validation MAE atau L1-loss equivalent karena fungsi loss training yang digunakan juga berbasis L1/MAE.

Grafik gabungan train_loss dan val_loss digunakan untuk melihat indikasi overfitting dan underfitting secara visual. Overfitting berat biasanya ditandai oleh train_loss yang terus menurun, tetapi val_loss justru meningkat dan gap antara validation loss dan training loss membesar tajam. Underfitting biasanya terlihat ketika train_loss dan val_loss sama-sama relatif tinggi serta tidak banyak membaik sepanjang epoch. Model yang lebih stabil umumnya menunjukkan train_loss dan val_loss yang sama-sama membaik lalu cenderung konvergen, dengan gap yang tidak membesar secara ekstrem.

Berdasarkan ringkasan 10 final run, rata-rata gap pada best epoch adalah {format_float(mean_gap_at_best)}, sedangkan rata-rata gap pada epoch terakhir adalah {format_float(mean_final_gap)}. Rata-rata minimum val_loss adalah {format_float(mean_min_val_loss)}, sedangkan rata-rata final val_loss adalah {format_float(mean_final_val_loss)}. Ringkasan diagnosis per run disimpan pada loss_diagnostic_summary_per_run.csv.

Perlu dicatat bahwa perbandingan train_loss dan val_loss pada notebook ini memiliki keterbatasan. Train_loss dihitung saat model berada dalam mode training, sedangkan val_loss berasal dari evaluasi validasi saat model berada dalam mode evaluasi. Oleh karena itu, train_loss dan val_loss tidak boleh diklaim sebagai perbandingan yang sepenuhnya apple-to-apple seperti train_eval_loss dan val_eval_loss yang sama-sama dihitung dalam mode evaluasi. Namun, kurva ini tetap berguna sebagai diagnostic post-hoc karena fungsi loss training dan metrik validasi yang digunakan sama-sama berbasis MAE/L1 pada skala target 0 sampai 1.

Kesimpulan umum: {general_conclusion}
""".strip()

interpretation_path = save_text(
    interpretation_text,
    OUT_POSTHOC / "overfit_underfit_interpretation.txt",
)

print(f"Narasi interpretasi disimpan ke: {interpretation_path}")
print()
print(interpretation_text)

Narasi interpretasi disimpan ke: C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\posthoc_diagnostics\overfit_underfit_interpretation.txt

Interpretasi Diagnostik Train Loss dan Validation Loss

Notebook pelengkap ini dibuat sebagai analisis post-hoc terhadap output eksperimen WavLM + LoRA pada strict split. Notebook ini tidak melakukan training ulang, tidak menjalankan Optuna ulang, dan tidak mengubah hasil eksperimen utama. Analisis dilakukan dengan membaca file history.csv dari setiap final run.

Pada notebook training utama, nilai train_loss dihitung menggunakan fungsi F.l1_loss(yhat, y, reduction="mean"). Dengan demikian, train_loss yang tersimpan pada history.csv merepresentasikan L1 loss atau MAE pada data train selama proses training. Sementara itu, metrik validasi dihitung melalui fungsi predict_and_evaluate(), yang menghasilkan val_mae_mean, val_rmse_mean, val_r2_mean, val_acc_mean, val_S, serta metrik per trait. Untuk kebutuhan diagnostik loss curve, no

In [14]:
# =========================
# BLOCK: STRICT SPLIT LEAKAGE AUDIT
# =========================

def normalize_split_value(value):
    value = str(value).strip().lower()
    
    mapping = {
        "train": "train",
        "training": "train",
        "tr": "train",
        "val": "val",
        "valid": "val",
        "validation": "val",
        "dev": "val",
        "test": "test",
        "testing": "test",
        "te": "test",
    }
    
    return mapping.get(value, value)


def build_overlap_rows(manifest_df: pd.DataFrame, split_col: str, id_col: str, id_type: str):
    rows = []
    splits = ["train", "val", "test"]
    
    split_sets = {}
    for split in splits:
        values = (
            manifest_df.loc[manifest_df[split_col] == split, id_col]
            .dropna()
            .astype(str)
            .unique()
        )
        split_sets[split] = set(values)
    
    pairs = [
        ("train", "val"),
        ("train", "test"),
        ("val", "test"),
    ]
    
    for left, right in pairs:
        overlap = sorted(split_sets[left].intersection(split_sets[right]))
        examples = ", ".join(overlap[:10])
        
        rows.append({
            "id_type": id_type,
            "left_split": left,
            "right_split": right,
            "overlap_count": len(overlap),
            "overlap_examples_first_10": examples,
        })
    
    return rows


leakage_audit_csv_path = OUT_POSTHOC / "strict_split_leakage_audit.csv"
leakage_audit_txt_path = OUT_POSTHOC / "strict_split_leakage_audit.txt"

if not MANIFEST_VAST.exists():
    leakage_text = f"""
Strict split leakage audit dilewati.

Alasan:
File manifest tidak ditemukan:
{MANIFEST_VAST}

Notebook tetap melanjutkan analisis loss diagnostic karena audit split bersifat tambahan.
    """.strip()
    
    leakage_audit_df = pd.DataFrame([{
        "audit_status": "skipped",
        "reason": "manifest_strict_vast.csv not found",
        "manifest_path": str(MANIFEST_VAST),
    }])
    
    save_csv(leakage_audit_df, leakage_audit_csv_path)
    save_text(leakage_text, leakage_audit_txt_path)
    
    print(leakage_text)

else:
    manifest_df = safe_read_csv(MANIFEST_VAST, required=True)
    
    split_col = pick_col(
        manifest_df,
        ["split_strict", "strict_split", "split", "dataset_split", "partition", "subset"],
    )
    
    clip_col = pick_col(
        manifest_df,
        ["clip_id", "clip", "video_id", "filename", "file_id"],
    )
    
    group_col = pick_col(
        manifest_df,
        ["group_id", "group", "speaker_id", "person_id", "video_group"],
    )
    
    audit_rows = []
    
    if split_col is None:
        audit_rows.append({
            "audit_status": "failed",
            "reason": "split column not found",
            "manifest_path": str(MANIFEST_VAST),
        })
        
        leakage_text = f"""
Strict split leakage audit gagal dilakukan.

Alasan:
Kolom split tidak ditemukan pada manifest:
{MANIFEST_VAST}

Kolom yang tersedia:
{list(manifest_df.columns)}
        """.strip()
    
    else:
        manifest_df = manifest_df.copy()
        manifest_df[split_col] = manifest_df[split_col].map(normalize_split_value)
        
        split_counts = manifest_df[split_col].value_counts().to_dict()
        
        if clip_col is not None:
            audit_rows.extend(
                build_overlap_rows(
                    manifest_df=manifest_df,
                    split_col=split_col,
                    id_col=clip_col,
                    id_type="clip_id",
                )
            )
        else:
            audit_rows.append({
                "id_type": "clip_id",
                "audit_status": "skipped",
                "reason": "clip_id column not found",
            })
        
        if group_col is not None:
            audit_rows.extend(
                build_overlap_rows(
                    manifest_df=manifest_df,
                    split_col=split_col,
                    id_col=group_col,
                    id_type="group_id",
                )
            )
        else:
            audit_rows.append({
                "id_type": "group_id",
                "audit_status": "skipped",
                "reason": "group_id column not found",
            })
        
        leakage_audit_df = pd.DataFrame(audit_rows)
        
        overlap_count_total = 0
        if "overlap_count" in leakage_audit_df.columns:
            overlap_count_total = pd.to_numeric(
                leakage_audit_df["overlap_count"],
                errors="coerce",
            ).fillna(0).sum()
        
        if overlap_count_total == 0:
            conclusion = (
                "Tidak ditemukan overlap clip_id atau group_id antar train, val, dan test "
                "berdasarkan kolom yang tersedia pada manifest."
            )
        else:
            conclusion = (
                "Ditemukan overlap antar split. Periksa strict_split_leakage_audit.csv "
                "untuk detail pasangan split dan contoh ID."
            )
        
        leakage_text = f"""
Strict Split Leakage Audit

Manifest:
{MANIFEST_VAST}

Kolom split yang digunakan:
{split_col}

Kolom clip yang digunakan:
{clip_col}

Kolom group yang digunakan:
{group_col}

Jumlah data per split:
{json.dumps(split_counts, indent=2, ensure_ascii=False)}

Kesimpulan:
{conclusion}
        """.strip()
    
    leakage_audit_df = pd.DataFrame(audit_rows)
    save_csv(leakage_audit_df, leakage_audit_csv_path)
    save_text(leakage_text, leakage_audit_txt_path)
    
    print(leakage_text)
    display(leakage_audit_df)

Strict Split Leakage Audit

Manifest:
C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\data\data\manifest_strict_vast.csv

Kolom split yang digunakan:
split_strict

Kolom clip yang digunakan:
clip_id

Kolom group yang digunakan:
group_id

Jumlah data per split:
{
  "train": 5936,
  "test": 2039,
  "val": 1999
}

Kesimpulan:
Tidak ditemukan overlap clip_id atau group_id antar train, val, dan test berdasarkan kolom yang tersedia pada manifest.


,id_type,left_split,right_split,overlap_count,overlap_examples_first_10
0,clip_id,train,val,0,
1,clip_id,train,test,0,
2,clip_id,val,test,0,
3,group_id,train,val,0,
4,group_id,train,test,0,
5,group_id,val,test,0,


In [15]:
# =========================
# BLOCK: OPTUNA REPRODUCIBILITY AUDIT NOTE
# =========================

def recursive_find_keys(obj, target_keys):
    found = []
    target_keys_lower = {str(k).lower() for k in target_keys}
    
    if isinstance(obj, dict):
        for key, value in obj.items():
            if str(key).lower() in target_keys_lower:
                found.append((key, value))
            found.extend(recursive_find_keys(value, target_keys))
    
    elif isinstance(obj, list):
        for item in obj:
            found.extend(recursive_find_keys(item, target_keys))
    
    return found


def recursive_find_text(obj, keywords):
    text = json.dumps(obj, ensure_ascii=False, default=str).lower()
    return {keyword: (keyword.lower() in text) for keyword in keywords}


optuna_note_path = OUT_POSTHOC / "optuna_reproducibility_note.txt"

if not OPTUNA_STUDY_CONFIG_PATH.exists():
    optuna_note = f"""
Optuna Reproducibility Audit

File study_config.json tidak ditemukan:
{OPTUNA_STUDY_CONFIG_PATH}

Audit ini dilewati karena konfigurasi Optuna tidak tersedia pada lokasi tersebut. Notebook ini tidak mengubah hasil eksperimen dan tidak menjalankan Optuna ulang.
    """.strip()

else:
    with open(OPTUNA_STUDY_CONFIG_PATH, "r", encoding="utf-8") as f:
        study_config = json.load(f)
    
    seed_matches = recursive_find_keys(
        study_config,
        target_keys=[
            "optuna_seed",
            "OPTUNA_SEED",
            "seed",
            "sampler_seed",
            "random_seed",
        ],
    )
    
    text_matches = recursive_find_text(
        study_config,
        keywords=[
            "TPESampler",
            "tpesampler",
            "sampler",
            "create_study",
            "OPTUNA_SEED",
            "optuna_seed",
        ],
    )
    
    if seed_matches:
        seed_lines = "\n".join([f"- {key}: {value}" for key, value in seed_matches])
        seed_status = f"Seed terkait Optuna ditemukan pada konfigurasi:\n{seed_lines}"
    else:
        seed_status = "Seed terkait Optuna tidak ditemukan secara eksplisit pada study_config.json."
    
    sampler_status = json.dumps(text_matches, indent=2, ensure_ascii=False)
    
    optuna_note = f"""
Optuna Reproducibility Audit

File konfigurasi:
{OPTUNA_STUDY_CONFIG_PATH}

{seed_status}

Pemeriksaan keyword terkait sampler:
{sampler_status}

Catatan interpretasi:
Notebook utama mencatat konfigurasi Optuna melalui study_config.json jika file tersebut tersedia. Namun, pencatatan optuna_seed saja belum otomatis menjamin reproducibility sampling Optuna secara penuh apabila create_study tidak secara eksplisit menggunakan sampler seperti TPESampler(seed=OPTUNA_SEED). Dengan kata lain, agar proses sampling Optuna benar-benar eksplisit dan lebih mudah direplikasi, sampler sebaiknya ditulis secara langsung, misalnya TPESampler(seed=OPTUNA_SEED), ketika study dibuat.

Audit ini hanya berupa catatan reproducibility. Notebook ini tidak mengubah hasil eksperimen utama, tidak menjalankan Optuna ulang, dan tidak mengganti konfigurasi training.
    """.strip()

save_text(optuna_note, optuna_note_path)

print(f"Catatan reproducibility Optuna disimpan ke: {optuna_note_path}")
print()
print(optuna_note)

Catatan reproducibility Optuna disimpan ke: C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\posthoc_diagnostics\optuna_reproducibility_note.txt

Optuna Reproducibility Audit

File konfigurasi:
C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\optuna_study\study_config.json

Seed terkait Optuna ditemukan pada konfigurasi:
- optuna_seed: 42

Pemeriksaan keyword terkait sampler:
{
  "TPESampler": false,
  "tpesampler": false,
  "sampler": false,
  "create_study": false,
  "OPTUNA_SEED": true,
  "optuna_seed": true
}

Catatan interpretasi:
Notebook utama mencatat konfigurasi Optuna melalui study_config.json jika file tersebut tersedia. Namun, pencatatan optuna_seed saja belum otomatis menjamin reproducibility sampling Optuna secara penuh apabila create_study tidak secara eksplisit menggunakan sampler seperti TPESampler(seed=OPTUNA_SEED). Dengan kata lain, agar proses sampling Optuna benar-benar eksplisit dan lebih mudah direplikasi, sampler sebai

In [16]:
# =========================
# BLOCK: COPY APPENDIX-READY OUTPUTS
# =========================

appendix_copy_plan = [
    (
        OUT_POSTHOC / "loss_diagnostic_summary_per_run.csv",
        OUT_APPENDIX / "appendix_loss_diagnostic_summary_per_run.csv",
    ),
    (
        OUT_POSTHOC / "aggregate_loss_curve_by_epoch.csv",
        OUT_APPENDIX / "appendix_aggregate_loss_curve_by_epoch.csv",
    ),
    (
        OUT_POSTHOC / "aggregate_train_vs_val_loss_mean_std.png",
        OUT_APPENDIX / "appendix_aggregate_train_vs_val_loss_mean_std.png",
    ),
    (
        OUT_POSTHOC / "aggregate_train_val_gap_mean_std.png",
        OUT_APPENDIX / "appendix_aggregate_train_val_gap_mean_std.png",
    ),
    (
        OUT_POSTHOC / "all_runs_train_vs_val_loss_overlay.png",
        OUT_APPENDIX / "appendix_all_runs_train_vs_val_loss_overlay.png",
    ),
    (
        OUT_POSTHOC / "overfit_underfit_interpretation.txt",
        OUT_APPENDIX / "appendix_overfit_underfit_interpretation.txt",
    ),
    (
        OUT_POSTHOC / "strict_split_leakage_audit.csv",
        OUT_APPENDIX / "appendix_strict_split_leakage_audit.csv",
    ),
    (
        OUT_POSTHOC / "strict_split_leakage_audit.txt",
        OUT_APPENDIX / "appendix_strict_split_leakage_audit.txt",
    ),
    (
        OUT_POSTHOC / "optuna_reproducibility_note.txt",
        OUT_APPENDIX / "appendix_optuna_reproducibility_note.txt",
    ),
]

APPENDIX_COPIED_FILES = []

for src, dst in appendix_copy_plan:
    src = Path(src)
    dst = Path(dst)
    
    if not src.exists():
        print(f"[SKIP] File appendix source tidak ditemukan: {src}")
        continue
    
    ensure_dir(dst.parent)
    shutil.copy2(src, dst)
    APPENDIX_COPIED_FILES.append(dst)
    print(f"[COPIED] {src} -> {dst}")

print(f"\nTotal file appendix copied: {len(APPENDIX_COPIED_FILES)}")

[COPIED] C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\posthoc_diagnostics\loss_diagnostic_summary_per_run.csv -> C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\appendix_ready\appendix_loss_diagnostic_summary_per_run.csv
[COPIED] C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\posthoc_diagnostics\aggregate_loss_curve_by_epoch.csv -> C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\appendix_ready\appendix_aggregate_loss_curve_by_epoch.csv
[COPIED] C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\posthoc_diagnostics\aggregate_train_vs_val_loss_mean_std.png -> C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\appendix_ready\appendix_aggregate_train_vs_val_loss_mean_std.png
[COPIED] C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\posthoc_diagnostics\aggregate_train_val_gap_mean_std.png -> C:\punya qq\tugas-akhir-qq\code\V2\workspace

In [17]:
# =========================
# BLOCK: FINAL NOTEBOOK OUTPUT SUMMARY
# =========================

posthoc_files = sorted([p for p in OUT_POSTHOC.glob("*") if p.is_file()])
appendix_files = sorted([p for p in OUT_APPENDIX.glob("appendix_*") if p.is_file()])

n_runs_read = len(RUN_HISTORIES)
n_total_epochs = len(all_runs_history_df)

avg_gap_best_epoch = loss_diagnostic_summary_df["gap_at_best_epoch"].mean()
avg_final_gap = loss_diagnostic_summary_df["final_gap"].mean()

overfit_like_count = loss_diagnostic_summary_df["diagnosis"].str.contains(
    "overfitting ringan", case=False, na=False
).sum()

underfit_like_count = loss_diagnostic_summary_df["diagnosis"].str.contains(
    "underfitting", case=False, na=False
).sum()

if overfit_like_count == 0:
    final_conclusion = (
        "Tidak terlihat indikasi overfitting berat secara umum berdasarkan gap train-validation "
        "dan pola loss post-hoc."
    )
else:
    final_conclusion = (
        f"Terdapat {overfit_like_count} run dengan indikasi overfitting ringan setelah epoch terbaik, "
        "tetapi interpretasi tetap perlu dilihat bersama mekanisme early stopping dan performa test akhir."
    )

if underfit_like_count > 0:
    final_conclusion += (
        f" Selain itu, terdapat {underfit_like_count} run dengan indikasi underfitting berdasarkan aturan sederhana."
    )

print("=" * 80)
print("POST-HOC LOSS DIAGNOSTICS FINISHED")
print("=" * 80)

print("\nFile yang dibuat di outputs/posthoc_diagnostics:")
for path in posthoc_files:
    print("-", path)

print("\nFile appendix yang tersedia/copy di outputs/appendix_ready:")
for path in appendix_files:
    print("-", path)

print("\nRingkasan singkat:")
print(f"- Jumlah run terbaca           : {n_runs_read}")
print(f"- Jumlah baris epoch total     : {n_total_epochs}")
print(f"- Rata-rata gap best epoch     : {format_float(avg_gap_best_epoch)}")
print(f"- Rata-rata final gap          : {format_float(avg_final_gap)}")
print(f"- Kesimpulan umum              : {final_conclusion}")

print("\nPreview summary per run:")
display(loss_diagnostic_summary_df)

POST-HOC LOSS DIAGNOSTICS FINISHED

File yang dibuat di outputs/posthoc_diagnostics:
- C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\posthoc_diagnostics\aggregate_loss_curve_by_epoch.csv
- C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\posthoc_diagnostics\aggregate_train_val_gap_mean_std.png
- C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\posthoc_diagnostics\aggregate_train_vs_val_loss_mean_std.png
- C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\posthoc_diagnostics\all_runs_history_with_val_loss.csv
- C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\posthoc_diagnostics\all_runs_train_vs_val_loss_overlay.png
- C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\posthoc_diagnostics\final_run_history_index.csv
- C:\punya qq\tugas-akhir-qq\code\V2\workspace_vastAI_v2\outputs\outputs\posthoc_diagnostics\invalid_history_files.txt
- C:\punya qq\tugas-akhi

,run_id,seed,best_epoch,stop_epoch,final_epoch,min_train_loss,min_val_loss,epoch_min_val_loss,train_loss_at_best_epoch,val_loss_at_best_epoch,gap_at_best_epoch,final_train_loss,final_val_loss,final_gap,mean_gap,max_gap,min_gap,diagnosis
0,run_01_seed_042,42,24,29,29,0.093535,0.098065,24,0.094404,0.098065,0.003661,0.093535,0.098203,0.004668,0.002612,0.004776,-0.003323,stabil / tidak menunjukkan overfitting berat; ...
1,run_02_seed_052,52,20,25,25,0.093958,0.098328,20,0.094986,0.098328,0.003343,0.093958,0.098632,0.004674,0.002177,0.004674,-0.004294,stabil / tidak menunjukkan overfitting berat; ...
2,run_03_seed_062,62,21,26,26,0.094005,0.098106,21,0.094607,0.098106,0.003500,0.094053,0.098644,0.004591,0.002348,0.005687,-0.003784,stabil / tidak menunjukkan overfitting berat; ...
3,run_04_seed_072,72,23,28,28,0.093795,0.098144,23,0.094336,0.098144,0.003808,0.093826,0.098176,0.004350,0.002477,0.005262,-0.003428,stabil / tidak menunjukkan overfitting berat; ...
4,run_05_seed_082,82,24,29,29,0.093684,0.098309,24,0.094085,0.098309,0.004224,0.093724,0.098406,0.004682,0.002551,0.004966,-0.003943,stabil / tidak menunjukkan overfitting berat; ...
5,run_06_seed_092,92,19,24,24,0.094179,0.098252,19,0.094785,0.098252,0.003467,0.094179,0.098791,0.004612,0.002258,0.005233,-0.003659,stabil / tidak menunjukkan overfitting berat; ...
6,run_07_seed_102,102,17,22,22,0.094217,0.098126,17,0.094730,0.098126,0.003397,0.094343,0.098465,0.004123,0.001873,0.004325,-0.004179,stabil / tidak menunjukkan overfitting berat; ...
7,run_08_seed_112,112,17,22,22,0.094291,0.098190,17,0.095250,0.098190,0.002940,0.094291,0.098677,0.004386,0.001900,0.004386,-0.004000,stabil / tidak menunjukkan overfitting berat; ...
8,run_09_seed_122,122,29,34,34,0.093362,0.098107,29,0.093732,0.098107,0.004375,0.093365,0.098485,0.005120,0.002961,0.005555,-0.003302,stabil / tidak menunjukkan overfitting berat; ...
9,run_10_seed_132,132,29,34,34,0.093282,0.098089,29,0.093886,0.098089,0.004203,0.093577,0.098120,0.004544,0.002735,0.005203,-0.004072,stabil / tidak menunjukkan overfitting berat; ...
